In [1]:
# ============================================================
# 1. Imports & Setup
# ============================================================
import warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import optuna

# Precision Settings
TORCH_DTYPE = torch.float32
NP_DTYPE = np.float32

torch.set_default_dtype(TORCH_DTYPE)
warnings.filterwarnings("ignore", category=FutureWarning)

# Device Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device in use: {device}")
if device.type == "cuda":
    print(f"CUDA device: {torch.cuda.current_device()} - {torch.cuda.get_device_name(0)}")

# ============================================================
# 2. Reproducibility & Data Loading
# ============================================================
base_seed = 2025
np.random.seed(base_seed)
torch.manual_seed(base_seed)
torch.cuda.manual_seed_all(base_seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Load Clean Reference
data_clean = np.load("lorenz96_100_noise_0.npy", allow_pickle=True).item()
l96 = data_clean["data"]
if l96.shape[0] == 100:  # Force (T, d) layout standard
    l96 = l96.T
l96 = l96.astype(NP_DTYPE)

# Load Noisy Dataset (10% global measurement noise added)
data_noisy = np.load("lorenz96_100_noise_10.npy", allow_pickle=True).item()
X_noisy_dataset = data_noisy["data"]
if X_noisy_dataset.shape[0] == 100:
    X_noisy_dataset = X_noisy_dataset.T
X_noisy_dataset = X_noisy_dataset.astype(NP_DTYPE)

X = torch.tensor(X_noisy_dataset, dtype=TORCH_DTYPE) 
X_true = torch.tensor(l96, dtype=TORCH_DTYPE)         

# Dataset Splits 
warmup_len, train_len, val_len, test_len = 1000, 399000, 50000, 50000
X_warmup = X[:warmup_len].to(device)
X_train= X[warmup_len:warmup_len + train_len].to(device)
X_val  = X[warmup_len + train_len : warmup_len + train_len + val_len].to(device)
X_test = X[warmup_len + train_len + val_len : warmup_len + train_len + val_len + test_len].to(device)

X_val_true  = X_true[warmup_len + train_len : warmup_len + train_len + val_len].to(device)
X_test_true = X_true[warmup_len + train_len + val_len : warmup_len + train_len + val_len + test_len].to(device)

# Structural Parameters
d = 100
horizons = [25, 50, 75, 100]
 

# ============================================================
# 3. Model Architecture & Helpers
# ============================================================
class FeatureMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim, dtype=TORCH_DTYPE),
            nn.Tanh(),
            nn.Linear(hidden_dim, output_dim, dtype=TORCH_DTYPE)
        )

    def forward(self, x):
        return self.net(x)


class AdaptiveNVARModel(nn.Module):
    def __init__(self, dk, m, d_out, hidden_dim):
        super().__init__()
        self.mlp = FeatureMLP(dk, hidden_dim, m)
        self.readout = nn.Linear(dk + m, d_out, bias=False, dtype=TORCH_DTYPE)

    def forward(self, H_lin):
        H_nn = self.mlp(H_lin)
        H_total = torch.cat([H_lin, H_nn], dim=1)
        return self.readout(H_total)


def construct_H_lin(X_tensor, k):
    """Build delay vectors: [x(t), x(t-1), ..., x(t-k+1)]"""
    T = X_tensor.shape[0]
    H = []
    for t in range(k - 1, T - 1):
        delays = [X_tensor[t - delay] for delay in range(k)]
        H.append(torch.cat(delays, dim=0))
    return torch.stack(H)


def init_weights_stable(m):
    if isinstance(m, nn.Linear):
        if m.bias is None:
            nn.init.normal_(m.weight, mean=0.0, std=1e-4)
        else:
            nn.init.xavier_normal_(m.weight, gain=nn.init.calculate_gain('tanh'))
            nn.init.zeros_(m.bias)


# ============================================================
# 4. Training Engine (Minibatch & State-to-State Configured)
# ============================================================
def train_joint_model(
    X_input, k, m, hidden_dim=200,
    lr_adam=1e-3, batch_size=1024, max_epochs_adam=50000,
    adam_patience=50, tolerance=1e-12, 
    device_target=None
):
    dev = device_target or device
    X_local = X_input.to(device=dev, dtype=TORCH_DTYPE)

    H_lin = construct_H_lin(X_local, k)       
    Y = X_local[k:]                           
    dk = H_lin.shape[1]
    d_out = Y.shape[1]
    num_samples = H_lin.shape[0]

    model = AdaptiveNVARModel(dk, m, d_out, hidden_dim).to(dev)
    model.apply(init_weights_stable)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr_adam, weight_decay=0.0)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=5
    )

    best_loss = float("inf")
    epochs_no_improve = 0
    in_memory_state = None
    
    for epoch in range(max_epochs_adam):
        model.train()
        
        indices = torch.randperm(num_samples, device=dev)
        epoch_loss = 0.0
        num_batches = 0
        
        for start_idx in range(0, num_samples, batch_size):
            batch_indices = indices[start_idx : start_idx + batch_size]
            
            H_batch = H_lin[batch_indices]
            Y_batch = Y[batch_indices]
            
            Y_hat = model(H_batch)
            loss = F.mse_loss(Y_hat, Y_batch)

            optimizer.zero_grad()
            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            epoch_loss += loss.item()
            num_batches += 1
            
        avg_epoch_loss = epoch_loss / num_batches
        scheduler.step(avg_epoch_loss)

        if best_loss - avg_epoch_loss > tolerance:
            best_loss = avg_epoch_loss
            epochs_no_improve = 0
            in_memory_state = {k_v: v.cpu().clone() for k_v, v in model.state_dict().items()}
        else:
            epochs_no_improve += 1

        if epochs_no_improve >= adam_patience:
            break

    if in_memory_state is not None:
        model.load_state_dict({k_v: v.to(dev) for k_v, v in in_memory_state.items()})

    return model


# ============================================================
# 5. Core Evaluation Pipeline Function (Overlapping Sliding Windows)
# ============================================================
def evaluate_model(
    model, k, X_true_target, X_history, horizon_max, stride, h_list, dev
):
    horizon_rmses = {h: [] for h in h_list}
    model.eval()

    total_len = len(X_true_target)
    
    for start_idx in range(k, total_len - horizon_max + 1, stride):
        y_true_window = X_true_target[start_idx : start_idx + horizon_max].to(device=dev, dtype=TORCH_DTYPE)
        X_init = X_history[start_idx - k : start_idx].to(device=dev, dtype=TORCH_DTYPE)
        
        x_t = [x.clone() for x in X_init.unbind(0)]
        H_lin = torch.cat([x_t[-1 - delay] for delay in range(k)], dim=0).unsqueeze(0)

        predictions = []
        for _ in range(horizon_max):
            with torch.no_grad():
                x_next = model(H_lin).squeeze(0)
            
            predictions.append(x_next)
            x_t = x_t[1:] + [x_next]
            H_lin = torch.cat([x_t[-1 - delay] for delay in range(k)], dim=0).unsqueeze(0)

        predictions_torch = torch.stack(predictions)

        for h in h_list:
            rmse = torch.sqrt(F.mse_loss(predictions_torch[:h], y_true_window[:h])).item()
            horizon_rmses[h].append(rmse)

    return {h: np.mean(horizon_rmses[h]) for h in h_list}


# ============================================================
# 6. Optuna Objective Function
# ============================================================
def objective(trial):
    k_suggest = trial.suggest_categorical("k", [2, 5, 10, 20])
    hidden_dim_suggest = trial.suggest_categorical("hidden_dim", [128, 1024])
    lr_adam_suggest = trial.suggest_categorical("lr_adam", [1e-4, 5e-4, 1e-3])
    
    # Suggest a value for 'c' dynamically (e.g., discrete integers from 1 to 20)
    c_suggest = trial.suggest_categorical("c", [2, 5, 10])
    m_suggest = c_suggest * d * k_suggest  
    
    validation_scores = []
    for eval_run in range(2): 
        run_seed = base_seed + eval_run
        torch.manual_seed(run_seed)
        torch.cuda.manual_seed_all(run_seed)
        
        model = train_joint_model(
            X_train, k_suggest, m_suggest, hidden_dim_suggest, lr_adam_suggest, device_target=device
        )
        
        rmse_by_horizon = evaluate_model(
            model=model, k=k_suggest, 
            X_true_target=X_val_true, X_history=X_val, 
            horizon_max=100, stride=10, h_list=horizons, dev=device
        )
        validation_scores.append(rmse_by_horizon[100])
        
    return np.mean(validation_scores)


# ============================================================
# 7. Main Automated Execution Workflow
# ============================================================
if __name__ == "__main__":
    print("Starting Automated Optuna Parameter Search Stage...\n")
    study = optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=base_seed)
    )
    study.optimize(objective, n_trials=20, show_progress_bar=True)

    best_trial = study.best_trial
    best_params = best_trial.params

    print("\n" + "=" * 60)
    print("BEST CONFIGURATION RETRIEVED AUTOMATICALLY")
    print("=" * 60)
    print(f"k          : {best_params['k']}")
    print(f"c          : {best_params['c']}")
    print(f"hidden_dim : {best_params['hidden_dim']}")
    print(f"lr_adam    : {best_params['lr_adam']:.6e}")
    print(f"Validation Target Cross-Seed Average RMSE@100: {best_trial.value:.6f}")

    print("\n=== Launching Final Test Benchmark Using Best Values ===")
    
    k_best = best_params['k']
    c_best = best_params['c']
    hd_best = best_params['hidden_dim']
    lr_best = best_params['lr_adam']
    m_best = c_best * d * k_best
    
    num_runs = 10
    all_run_rmses = {h: [] for h in horizons}

    for run in range(num_runs):
        run_seed = base_seed + run
        torch.manual_seed(run_seed)
        torch.cuda.manual_seed_all(run_seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

        model = train_joint_model(X_train, k=k_best, m=m_best, hidden_dim=hd_best, lr_adam=lr_best, device_target=device)
        
        rmse_by_horizon = evaluate_model(
            model=model, k=k_best, 
            X_true_target=X_test_true, X_history=X_test, 
            horizon_max=100, stride=10, h_list=horizons, dev=device
        )
    
        for h in horizons:
            all_run_rmses[h].append(rmse_by_horizon[h])
        print(f"Run {run+1:02d}/{num_runs} Complete. Seed: {run_seed} -> h100: {rmse_by_horizon[100]:.6f}")

    # --- PHASE 3: FINAL SCIENTIFIC PRESENTATION REPORT ---
    final_stats = {h: (np.mean(all_run_rmses[h]), np.std(all_run_rmses[h], ddof=1)) for h in horizons}

    print("\n" + "=" * 60)
    print("FINAL TEST EXTRAPOLATION SUMMARY (Over 10 Runs)")
    print("=" * 60)
    print(f"Parameters utilized: k={k_best}, c={c_best}, hidden_dim={hd_best}, Training Data: Noisy Dataset (0%)")
    print("-" * 60)
    for h in horizons:
        mean, std = final_stats[h]
        print(f"Horizon {h:3d} steps: {mean:.6f} ± {std:.6f}")

Device in use: cuda
CUDA device: 0 - A100-PCIE-40GB


[I 2026-07-05 22:03:03,056] A new study created in memory with name: no-name-6f8840c6-2629-4eba-bf80-4a666e3a6c3b


Starting Automated Optuna Parameter Search Stage...



  0%|          | 0/20 [00:00<?, ?it/s]

[I 2026-07-05 22:18:39,493] Trial 0 finished with value: 0.8025774995525758 and parameters: {'k': 10, 'hidden_dim': 128, 'lr_adam': 0.001, 'c': 10}. Best is trial 0 with value: 0.8025774995525758.
[I 2026-07-05 22:40:54,242] Trial 1 finished with value: 0.8755769190066802 and parameters: {'k': 5, 'hidden_dim': 1024, 'lr_adam': 0.001, 'c': 2}. Best is trial 0 with value: 0.8025774995525758.
[I 2026-07-05 22:56:10,093] Trial 2 finished with value: 0.8448773221107189 and parameters: {'k': 5, 'hidden_dim': 128, 'lr_adam': 0.0005, 'c': 2}. Best is trial 0 with value: 0.8025774995525758.
[I 2026-07-05 23:07:40,641] Trial 3 finished with value: 0.8026228543035014 and parameters: {'k': 10, 'hidden_dim': 128, 'lr_adam': 0.0005, 'c': 2}. Best is trial 0 with value: 0.8025774995525758.
[I 2026-07-05 23:27:53,579] Trial 4 finished with value: 0.761621449957345 and parameters: {'k': 20, 'hidden_dim': 1024, 'lr_adam': 0.001, 'c': 5}. Best is trial 4 with value: 0.761621449957345.
[I 2026-07-06 00:20